# RFM 会员价值分析项目

**项目类型：** Python 数据分析 / 会员分群 / 数据可视化  
**技术栈：** Python、Pandas、NumPy、Pyecharts、Jupyter Notebook；MySQL 为可选扩展。

## 1. 项目背景

企业积累订单数据后，需要从历史交易行为中识别不同价值层级的会员。本项目使用 RFM 模型，以最近一次消费时间、消费频率和消费金额三个维度构建会员年度画像，并对会员分群结构进行年度比较。

本 Notebook 将原始案例整理为适合学习、复现和 GitHub 作品集展示的项目流程：数据读取 → 数据质量检查 → 清洗 → RFM 指标构建 → 评分分群 → 可视化 → 结果导出。

## 2. 分析目标

1. 读取 2015—2018 年订单数据并检查字段、类型与缺失情况。
2. 清理必要字段缺失记录、筛选有效订单。
3. 按“年份 + 会员 ID”计算 R、F、M 三项指标。
4. 按设定区间对 RFM 指标进行三档评分，形成三位分群编码。
5. 比较年度会员记录数与各 RFM 分群的结构变化。
6. 导出 CSV、Excel 和可在浏览器中打开的交互式 HTML 图表。
7. 将数据库导出作为可选功能，避免它阻塞核心分析流程。

## 3. RFM 指标定义

| 指标 | 全称 | 本项目口径 | 评分方向 |
|---|---|---|---|
| R | Recency | 年度统计节点距离会员最近一次消费的天数 | 天数越少，得分越高 |
| F | Frequency | 该会员该年度订单号非空记录数 | 次数越多，得分越高 |
| M | Monetary | 该会员该年度订单金额总和 | 金额越高，得分越高 |

例如，`333` 表示三项评分均为 3；`111` 表示三项评分均为 1。分数只代表当前评分规则下的分层，不是行业通用标准。

## 4. 数据口径与注意事项

- 每年的统计节点采用该年度清洗后订单中的最大提交日期，不一定是自然年 12 月 31 日。
- 按“年份 + 会员 ID”统计，因此同一会员可能在多个年度重复出现；年度会员记录数不是跨年去重人数。
- `F` 按订单号非空记录数计算。如果一张订单在源表中对应多条商品明细，应先确认数据粒度，必要时改为订单号去重计数。
- 分箱阈值沿用原始案例中的设定，正式业务分析时应结合数据分布和业务目标复核。
- RFM 描述历史交易特征，分群变化本身不能证明某项营销活动造成了因果结果。

## 5. 项目目录与数据准备

将 Excel 文件放到 `data/sales.xlsx`，也可以放在 Notebook 同级目录并命名为 `sales.xlsx`。

```text
RFM_GitHub_Project/
├── RFM会员价值分析.ipynb
├── README.md
├── requirements.txt
├── .gitignore
├── data/
│   └── sales.xlsx       # 本地使用，不提交到公开仓库
└── outputs/             # 运行后生成的结果，默认不提交
```

Excel 需要包含 `2015`、`2016`、`2017`、`2018` 四个工作表。每个工作表至少包含 `提交日期`、`会员ID`、`订单号`、`订单金额` 四个字段。原案例中的 `会员等级` 表不参与主分析流程。

**隐私提醒：** 不要将未经授权的原始交易数据、真实会员标识、数据库密码或其他敏感信息上传到 GitHub。

## 6. 环境准备

建议使用 Python 3.10—3.12。进入项目目录后执行：

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
jupyter notebook
```

Windows 激活虚拟环境的命令为：

```powershell
.venv\Scripts\activate
```

运行时请从上到下依次执行单元格。如果缺少数据文件，代码会在读取阶段给出提示。

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

from pyecharts.charts import Bar, Line, Pie
import pyecharts.options as opts

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = DATA_DIR / "sales.xlsx"
if not DATA_FILE.exists() and (PROJECT_DIR / "sales.xlsx").exists():
    DATA_FILE = PROJECT_DIR / "sales.xlsx"

ORDER_SHEETS = ["2015", "2016", "2017", "2018"]
REQUIRED_COLUMNS = ["提交日期", "会员ID", "订单号", "订单金额"]

# 沿用原案例阈值；如果实际数据超出最后一个上界，后续会扩展上界。
R_BINS_BASE = [-1, 79, 255, 365]
F_BINS_BASE = [0, 2, 5, 130]
M_BINS_BASE = [1, 69, 1199, 206252]

print("项目目录：", PROJECT_DIR.resolve())
print("数据文件：", DATA_FILE.resolve())
print("输出目录：", OUTPUT_DIR.resolve())

## 7. 加载数据与基础检查

首先读取四个年度工作表。随后检查每个工作表的行列数、必需字段、样例记录和缺失值情况。此步骤的目的是尽早发现字段名不一致、日期格式异常或数据缺失等问题。

In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"未找到数据文件：{DATA_FILE}\n"
        "请将有权使用的 Excel 文件放入 data/sales.xlsx，或放在 Notebook 同级目录并命名为 sales.xlsx。"
    )

raw_sheets = pd.read_excel(DATA_FILE, sheet_name=ORDER_SHEETS)

print("读取到的工作表：", list(raw_sheets.keys()))
for sheet_name, df in raw_sheets.items():
    missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing_columns:
        raise ValueError(f"工作表 {sheet_name} 缺少字段：{missing_columns}")

    print(f"\n工作表：{sheet_name}，数据形状：{df.shape}")
    display(df.head(3))
    display(df[REQUIRED_COLUMNS].isna().sum().rename("缺失值数量").to_frame())

## 8. 数据清洗

本项目的清洗规则：

1. 将 `提交日期` 转换为日期类型；无法解析的日期变为缺失值。
2. 将 `订单金额` 转为数值类型；无法转换的内容变为缺失值。
3. 仅删除分析必需字段存在缺失的记录，避免因其他无关字段为空而整行删除。
4. 过滤 `订单金额 > 1` 的订单，沿用原案例的业务规则。
5. 将每个年度清洗后订单中的最大提交日期设为该年度统计节点。

注意：该统计节点是该年数据中的最后交易日期，并不保证是 12 月 31 日。每一步删除的数据量会记录在质量报告中。

In [ ]:
clean_sheets = {}
quality_rows = []

for year_sheet in ORDER_SHEETS:
    df = raw_sheets[year_sheet].copy()
    original_rows = len(df)

    df["提交日期"] = pd.to_datetime(df["提交日期"], errors="coerce")
    df["订单金额"] = pd.to_numeric(df["订单金额"], errors="coerce")

    before_filter = len(df)
    df = df.dropna(subset=REQUIRED_COLUMNS)
    after_required_filter = len(df)

    df = df[df["订单金额"] > 1].copy()
    after_amount_filter = len(df)

    if df.empty:
        raise ValueError(f"{year_sheet} 年清洗后没有可用订单，请检查数据和过滤条件。")

    df["year"] = int(year_sheet)
    df["reference_date"] = df["提交日期"].max()
    clean_sheets[year_sheet] = df

    quality_rows.append({
        "年份": int(year_sheet),
        "原始行数": original_rows,
        "必要字段缺失或无法解析删除数": before_filter - after_required_filter,
        "金额规则过滤删除数": after_required_filter - after_amount_filter,
        "清洗后行数": after_amount_filter,
        "年度统计节点": df["reference_date"].iloc[0]
    })

quality_report = pd.DataFrame(quality_rows)
display(quality_report)

## 9. 合并年度订单并计算 R、F、M

将四个年度的数据纵向合并，再按 `year` 和 `会员ID` 分组：

- `r`：统计节点与最近一次消费日期相差的天数，取最小值。
- `f`：订单号非空记录数。
- `m`：订单金额总和。

完成后先查看结果样例和描述性统计，检查是否存在明显异常。

In [ ]:
df_merge = pd.concat([clean_sheets[y] for y in ORDER_SHEETS], ignore_index=True)
df_merge["date_interval"] = (df_merge["reference_date"] - df_merge["提交日期"]).dt.days

rfm_gb = (
    df_merge.groupby(["year", "会员ID"], as_index=False)
    .agg(
        r=("date_interval", "min"),
        f=("订单号", "count"),
        m=("订单金额", "sum")
    )
)

print("合并后订单记录数：", len(df_merge))
print("年度-会员组合数：", len(rfm_gb))
display(rfm_gb.head())
display(rfm_gb[["r", "f", "m"]].describe().T)

## 10. 将 R、F、M 转换为三档评分

评分逻辑：

- R：消费间隔越短越好，因此评分顺序为 3、2、1。
- F：消费次数越多越好，因此评分顺序为 1、2、3。
- M：消费金额越高越好，因此评分顺序为 1、2、3。

阈值沿用原案例：R 为 `[-1, 79, 255, 365]`，F 为 `[0, 2, 5, 130]`，M 为 `[1, 69, 1199, 206252]`。如果数据最大值超过原区间上界，代码只扩展最后一个上界，避免记录因越界而没有评分。实际项目应根据数据分布和业务要求复核阈值。

In [ ]:
def extend_upper_bound(base_bins, observed_max):
    bins = list(base_bins)
    if observed_max >= bins[-1]:
        bins[-1] = float(observed_max) + 1
    return bins

r_bins = extend_upper_bound(R_BINS_BASE, rfm_gb["r"].max())
f_bins = extend_upper_bound(F_BINS_BASE, rfm_gb["f"].max())
m_bins = extend_upper_bound(M_BINS_BASE, rfm_gb["m"].max())

rfm_gb["r_label"] = pd.cut(
    rfm_gb["r"], bins=r_bins, labels=[3, 2, 1], include_lowest=True
)
rfm_gb["f_label"] = pd.cut(
    rfm_gb["f"], bins=f_bins, labels=[1, 2, 3], include_lowest=True
)
rfm_gb["m_label"] = pd.cut(
    rfm_gb["m"], bins=m_bins, labels=[1, 2, 3], include_lowest=True
)

label_cols = ["r_label", "f_label", "m_label"]
unscored = rfm_gb[label_cols].isna().any(axis=1)
print("未完成评分的记录数：", int(unscored.sum()))
if unscored.any():
    display(rfm_gb.loc[unscored].head(10))
    raise ValueError("存在未完成评分的记录，请检查数据或分箱阈值。")

rfm_gb[label_cols] = rfm_gb[label_cols].astype(int).astype(str)
rfm_gb["rfm_group"] = (
    rfm_gb["r_label"] + rfm_gb["f_label"] + rfm_gb["m_label"]
)
display(rfm_gb.head())
display(rfm_gb["rfm_group"].value_counts().rename_axis("RFM分群").to_frame("会员记录数"))

## 11. 数据校验

可视化前进行基本断言：

- 每个“年份—会员 ID”组合只出现一次。
- R、F、M 指标满足预期的非负或正值约束。
- 每条记录都生成三位数字的 RFM 分群编码。

如果校验失败，应先检查数据质量或评分区间，而不是忽略异常继续分析。

In [ ]:
assert not rfm_gb.duplicated(["year", "会员ID"]).any()
assert (rfm_gb["r"] >= 0).all()
assert (rfm_gb["f"] >= 1).all()
assert (rfm_gb["m"] > 1).all()
assert rfm_gb["rfm_group"].str.fullmatch(r"[1-3]{3}").all()
print("数据校验通过。")

## 12. 汇总年度分群统计

按 `rfm_group` 和 `year` 统计会员记录数，同时生成年度总体指标表。年度分群统计可用于查看会员结构变化，但不能直接说明变化由某项营销活动造成。

In [ ]:
display_data = (
    rfm_gb.groupby(["rfm_group", "year"], as_index=False)
    .agg(number=("会员ID", "count"))
    .sort_values(["year", "rfm_group"])
)

year_summary = (
    rfm_gb.groupby("year", as_index=False)
    .agg(
        会员记录数=("会员ID", "count"),
        R平均值=("r", "mean"),
        F平均值=("f", "mean"),
        M平均值=("m", "mean"),
        分群数量=("rfm_group", "nunique")
    )
)

display(year_summary.round(2))
display(display_data.pivot(index="rfm_group", columns="year", values="number").fillna(0).astype(int))

## 13. 可视化一：年度会员记录数趋势

折线图比较不同年度的会员记录数。这里统计的是“年度会员记录”，不是跨年度去重后的独立会员人数。

In [ ]:
year_chart = (
    Line()
    .add_xaxis(year_summary["year"].astype(str).tolist())
    .add_yaxis(
        "年度会员记录数",
        year_summary["会员记录数"].astype(int).tolist(),
        is_smooth=True,
        label_opts=opts.LabelOpts(is_show=True)
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="2015—2018 年度会员记录数趋势"),
        xaxis_opts=opts.AxisOpts(name="年份"),
        yaxis_opts=opts.AxisOpts(name="会员记录数"),
        tooltip_opts=opts.TooltipOpts(trigger="axis")
    )
)
year_chart.render(str(OUTPUT_DIR / "01_年度会员记录数趋势.html"))
year_chart.render_notebook()

## 14. 可视化二：不同年度的 RFM 分群人数对比

柱状图对比各 RFM 分群在不同年份的记录数。为保证年份之间可比较，代码会把缺失的“分群—年份”组合补为 0。

In [ ]:
years = sorted(display_data["year"].unique().tolist())
groups = sorted(display_data["rfm_group"].unique().tolist())

full_index = pd.MultiIndex.from_product([groups, years], names=["rfm_group", "year"])
plot_df = (
    display_data.set_index(["rfm_group", "year"])["number"]
    .reindex(full_index, fill_value=0)
    .reset_index()
)

group_chart = Bar().add_xaxis(groups)
for year in years:
    values = (
        plot_df.loc[plot_df["year"] == year]
        .set_index("rfm_group").reindex(groups)["number"]
        .fillna(0).astype(int).tolist()
    )
    group_chart.add_yaxis(str(year), values)

group_chart.set_global_opts(
    title_opts=opts.TitleOpts(title="各年度 RFM 分群人数对比"),
    xaxis_opts=opts.AxisOpts(name="RFM 分群", axislabel_opts=opts.LabelOpts(rotate=45)),
    yaxis_opts=opts.AxisOpts(name="会员记录数"),
    tooltip_opts=opts.TooltipOpts(trigger="axis"),
    datazoom_opts=[opts.DataZoomOpts(type_="slider")]
)
group_chart.render(str(OUTPUT_DIR / "02_RFM分群人数对比.html"))
group_chart.render_notebook()

## 15. 可视化三：最近一年 RFM 分群占比

饼图展示最近一年各分群在该年度会员记录中的占比。它用于观察分群结构，不应单独作为因果判断依据。

In [ ]:
latest_year = int(max(years))
latest_data = (
    display_data[display_data["year"] == latest_year]
    .sort_values("number", ascending=False)
)

pie_chart = (
    Pie()
    .add(
        "会员记录数",
        [(str(row.rfm_group), int(row.number)) for row in latest_data.itertuples(index=False)],
        radius=["35%", "65%"],
        rosetype="radius"
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title=f"{latest_year} 年 RFM 分群占比"),
        legend_opts=opts.LegendOpts(orient="vertical", pos_left="left"),
        tooltip_opts=opts.TooltipOpts(formatter="{b}: {c} ({d}%)")
    )
)
pie_chart.render(str(OUTPUT_DIR / "03_最近一年RFM分群占比.html"))
pie_chart.render_notebook()

## 16. 分群变化与业务观察

输出每年人数最多的分群，并比较首尾年份分群人数的变化。变化率在基期人数为 0 时记为缺失，避免除零。

In [ ]:
top_group_each_year = (
    display_data.sort_values(["year", "number"], ascending=[True, False])
    .groupby("year", as_index=False).first()
    .rename(columns={"rfm_group": "人数最多的分群", "number": "该分群会员记录数"})
)
display(top_group_each_year)

first_year, last_year = min(years), max(years)
comparison = (
    display_data[display_data["year"].isin([first_year, last_year])]
    .pivot(index="rfm_group", columns="year", values="number")
    .fillna(0)
)
comparison.columns = [str(c) for c in comparison.columns]
comparison["人数变化"] = comparison[str(last_year)] - comparison[str(first_year)]
comparison["变化率(%)"] = np.where(
    comparison[str(first_year)] > 0,
    comparison["人数变化"] / comparison[str(first_year)] * 100,
    np.nan
)
comparison = comparison.sort_values("人数变化", ascending=False)

print(f"{first_year}—{last_year} 年增加最多的分群：")
display(comparison.head(10).round(2))
print(f"{first_year}—{last_year} 年减少最多的分群：")
display(comparison.tail(10).sort_values("人数变化").round(2))

## 17. 导出分析结果

所有导出文件都保存到 `outputs/`，避免写入个人电脑的绝对路径。CSV 使用 `utf-8-sig`，方便在部分 Excel 环境中查看中文。

In [ ]:
rfm_gb.to_csv(OUTPUT_DIR / "rfm_member_scores.csv", index=False, encoding="utf-8-sig")
display_data.to_csv(OUTPUT_DIR / "rfm_group_year_summary.csv", index=False, encoding="utf-8-sig")
year_summary.to_csv(OUTPUT_DIR / "year_summary.csv", index=False, encoding="utf-8-sig")

with pd.ExcelWriter(OUTPUT_DIR / "RFM分析结果.xlsx", engine="openpyxl") as writer:
    rfm_gb.to_excel(writer, sheet_name="会员RFM评分", index=False)
    display_data.to_excel(writer, sheet_name="年度分群统计", index=False)
    year_summary.to_excel(writer, sheet_name="年度总体统计", index=False)

print("分析结果已输出到：", OUTPUT_DIR.resolve())

## 18. 可选：导出到 MySQL

MySQL 不是完成 RFM 分析的必需条件。只有在数据库已启动、`rfm_db` 已创建且连接配置正确时再运行。密码不要硬编码到 Notebook 或提交到公开仓库。

在终端设置环境变量后，可取消下方代码注释：

```bash
export RFM_MYSQL_URL='mysql+pymysql://用户名:密码@localhost:3306/rfm_db?charset=utf8mb4'
```

In [ ]:
# 可选扩展：需要 MySQL 时再启用
# import os
# from sqlalchemy import create_engine
#
# mysql_url = os.getenv("RFM_MYSQL_URL")
# if not mysql_url:
#     raise ValueError("请先设置环境变量 RFM_MYSQL_URL。")
#
# engine = create_engine(mysql_url, pool_pre_ping=True)
# rfm_gb.to_sql("rfm_table", engine, index=False, if_exists="replace")
# print("已写入 MySQL 表 rfm_table")

## 19. 结论、业务建议与局限性

### 业务解读参考

- `333`：R、F、M 三项评分都高，可作为高价值会员候选群体，进一步分析其复购、留存和服务需求。
- `3xx`：最近消费较近，可结合 F、M 判断其活跃度与消费贡献。
- `1xx`：最近消费间隔较长，可以进一步检查活跃度，但不能仅凭这一项直接认定会员流失。
- `x1x`：消费频率评分较低，可以观察购买周期、品类偏好和二次购买机会。
- `xx1`：金额评分较低，可以进一步分析客单价、商品结构和促销响应。

### 项目局限

1. 统计节点是每年清洗后订单中的最大提交日期，不一定是自然年末。
2. RFM 仅使用历史交易数据，无法解释会员不购买的原因。
3. 分群依赖评分阈值；阈值变化会影响分群结果。
4. `F` 当前按订单号非空记录数计算。如果一张订单对应多条商品明细，可能高估订单频率，应根据源数据粒度考虑去重。
5. 分群人数变化只能说明结构变化，不能单独证明某项营销措施带来了因果影响。

## 20. 项目复现检查清单

- [ ] 已将 `sales.xlsx` 放入 `data/` 或项目根目录。
- [ ] 已安装 `requirements.txt` 中的依赖。
- [ ] 已按顺序运行全部必要单元格。
- [ ] 已检查数据清洗报告和 RFM 分群结果。
- [ ] 已打开 `outputs/` 中生成的 HTML 图表。
- [ ] 上传 GitHub 前已移除原始数据、会员信息、密码和敏感输出。